# ConFit Fine-Tuning — Notebook 3: Synthetic Pair Generation & Training

Takes the already-extracted `jd_requirements.jsonl` as the **single source of truth** and uses `gpt-oss-20b` (no reasoning/CoT) to generate a synthetic **(Positive, Hard Negative)** pair for every JD requirement sentence.

| Step | What | Output |
|------|------|--------|
| 1 | Load JD requirements from disk | flat list of skill rows |
| 2 | LLM generates Positive + Hard Negative per requirement (8 semaphores, incremental save) | `synthetic_pairs.jsonl` |
| 3 | Inspect & QA generated pairs | stats DataFrame |
| 4 | Build `InputExample` triplets (anchor, positive, negative) | train / val split |
| 5 | Fine-tune `multilingual-e5-small` with TripletLoss | `models/confit-v1/` |

**Strategy:** The JD *requirement* sentence is the **Anchor**. The LLM writes:
- **Positive** — a real-sounding resume bullet that perfectly proves the skill at the stated level
- **Hard Negative** — a resume bullet that uses almost-identical keywords but *fails* to prove the skill (wrong depth, keyword stuffing, passive involvement)

This eliminates all dependency on O*NET matching or cross-dataset alignment.

In [ ]:
%pip install -q sentence-transformers torch datasets nest_asyncio tqdm

In [ ]:
import sys
import os
import json
import re
import asyncio
import warnings
from pathlib import Path
from datetime import datetime

import nest_asyncio
import pandas as pd
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")

# ── Project root so we can import app/ modules ────────────────────────────────
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

nest_asyncio.apply()

# ── I/O Paths ─────────────────────────────────────────────────────────────────
NB2_OUT_DIR   = NOTEBOOK_DIR / "data" / "nb2_outputs"
NB3_OUT_DIR   = NOTEBOOK_DIR / "data" / "nb3_outputs"
NB3_OUT_DIR.mkdir(parents=True, exist_ok=True)

JD_INPUT_FILE    = NB2_OUT_DIR / "jd_requirements.jsonl"  # ← from NB2
PAIRS_FILE       = NB3_OUT_DIR / "synthetic_pairs.jsonl"  # ← incremental output
MODEL_OUTPUT_DIR = NOTEBOOK_DIR / "models" / "confit-v1"
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Concurrency ───────────────────────────────────────────────────────────────
MAX_SEMAPHORES = 8   # 8 parallel LLM calls at once

print(f"JD input  : {JD_INPUT_FILE}")
print(f"Pairs out : {PAIRS_FILE}")
print(f"Model out : {MODEL_OUTPUT_DIR}")

In [ ]:
# ── Import the same production LLM client used in NB2 ────────────────────────
from app.clients.nvidia_llm_client import nvidia_llm_client
from app.config import settings

print(f"Model : {nvidia_llm_client.model}")
print(f"API   : {settings.NVIDIA_BASE_URL[:50]}...")

# ── No-reasoning LLM call ─────────────────────────────────────────────────────
# gpt-oss-20b supports disabling CoT via chat_template_kwargs {"thinking": False}.
# We also only read .content (not .reasoning_content) as a safe fallback.
async def _call_llm_no_reasoning(system: str, user: str) -> str:
    """
    Calls gpt-oss-20b with reasoning/CoT explicitly disabled.
    Falls back gracefully if the extra kwarg is unsupported by the endpoint.
    """
    try:
        completion = await nvidia_llm_client.client.chat.completions.create(
            model=nvidia_llm_client.model,
            messages=[
                {"role": "system", "content": system},
                {"role": "user",   "content": user},
            ],
            temperature=0.2,   # slight diversity across pairs
            stream=False,
            extra_body={"chat_template_kwargs": {"thinking": False}},  # disable CoT
        )
    except Exception:
        # Fallback — model may not support the extra kwarg
        completion = await nvidia_llm_client.client.chat.completions.create(
            model=nvidia_llm_client.model,
            messages=[
                {"role": "system", "content": system},
                {"role": "user",   "content": user},
            ],
            temperature=0.2,
            stream=False,
        )
    return (completion.choices[0].message.content or "").strip()

print("LLM helper defined — no-reasoning mode via chat_template_kwargs.")

In [ ]:
# ── Section 2: Load all skill rows from jd_requirements.jsonl ────────────────
# Each row = one (jd_index, job_title, skill, requirement, level)
# This is the ANCHOR dataset — no cross-dataset matching needed.

all_rows = []
with open(JD_INPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        for s in rec.get("skills", []):
            req = s.get("requirement", "").strip()
            if req:  # skip empty requirements
                all_rows.append({
                    "jd_index":    rec["jd_index"],
                    "job_title":   rec["job_title"],
                    "skill":       s.get("skill", "").strip(),
                    "requirement": req,
                    "level":       s.get("level", "mid"),
                })

df_all = pd.DataFrame(all_rows)
print(f"Total skill requirement rows loaded : {len(all_rows):,}")
print(f"Unique JDs                          : {df_all['jd_index'].nunique():,}")
print(f"\nLevel distribution:")
print(df_all["level"].value_counts().to_string())
print()
display(df_all.head(5))

In [ ]:
# ── Resumability — skip already-processed (jd_index, skill) combos ───────────
processed_keys: set = set()
if PAIRS_FILE.exists():
    with open(PAIRS_FILE, "r", encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
                processed_keys.add((obj["jd_index"], obj["skill"]))
            except Exception:
                pass
    print(f"Resuming: {len(processed_keys):,} rows already processed.")
else:
    print("Starting fresh — no existing pairs file.")

pending_rows = [
    r for r in all_rows
    if (r["jd_index"], r["skill"]) not in processed_keys
]
print(f"Pending  : {len(pending_rows):,}  |  Already done: {len(processed_keys):,}")

In [ ]:
# ── Section 3: Pair Generation Prompt ────────────────────────────────────────

PAIR_SYSTEM = (
    "You are a training data generator for a recruitment AI. "
    "Output ONLY a valid JSON object. No explanation, no reasoning, no markdown fences. "
    "Respond immediately with the JSON — no preamble."
)

PAIR_PROMPT = """\
Given this job requirement:
  Skill: {skill}
  Level: {level}
  Requirement: "{requirement}"

Generate ONE training pair:
1. positive     — a realistic resume bullet that PERFECTLY proves this skill at the stated level.
2. hard_negative — a resume bullet that uses SIMILAR keywords but clearly FAILS to prove
                   the skill (e.g., too shallow, wrong context, or passive/vague involvement).

Return ONLY this JSON object — nothing else:
{{"positive": "<resume bullet>", "hard_negative": "<tricky mismatch bullet>"}}

Rules:
- positive must use strong action verbs and concrete outcomes (numbers/scale if possible)
- hard_negative must LOOK relevant but reveal lack of depth on close reading
- Both must be written as resume first-person bullets (action verb first)
- Maximum 2 sentences each
"""

# ── JSON extraction helper ─────────────────────────────────────────────────────
_OBJ_RE = re.compile(r"\{.*\}", re.DOTALL)

def _extract_pair(raw: str) -> dict | None:
    """Pull {positive, hard_negative} dict from LLM reply."""
    if not raw:
        return None
    raw = re.sub(r"```(?:json)?", "", raw).strip().strip("`").strip()
    try:
        obj = json.loads(raw)
        if isinstance(obj, dict) and "positive" in obj and "hard_negative" in obj:
            return obj
    except Exception:
        pass
    m = _OBJ_RE.search(raw)
    if m:
        try:
            obj = json.loads(m.group())
            if isinstance(obj, dict) and "positive" in obj and "hard_negative" in obj:
                return obj
        except Exception:
            pass
    return None

print("Prompt and JSON extraction helper defined.")

In [ ]:
# ── Async task: one skill row → one JSONL record ──────────────────────────────
async def _generate_one_pair(sem: asyncio.Semaphore, row: dict) -> dict:
    """Calls LLM to generate (positive, hard_negative) for a single JD skill requirement."""
    async with sem:
        try:
            raw_reply = await _call_llm_no_reasoning(
                system=PAIR_SYSTEM,
                user=PAIR_PROMPT.format(
                    skill=row["skill"],
                    level=row["level"],
                    requirement=row["requirement"],
                ),
            )
        except Exception as e:
            print(f"\n[!] LLM failed for ({row['jd_index']}, {row['skill']}): {e}")
            raw_reply = ""

    pair = _extract_pair(raw_reply)
    return {
        "jd_index":      row["jd_index"],
        "job_title":     row["job_title"],
        "skill":         row["skill"],
        "level":         row["level"],
        "anchor":        row["requirement"],       # ← original JD sentence
        "positive":      pair["positive"]       if pair else None,
        "hard_negative": pair["hard_negative"]  if pair else None,
        "generated_at":  datetime.utcnow().isoformat(),
        "status":        "ok" if pair else "failed",
    }


# ── Async orchestrator — 8 semaphores, incremental flush ─────────────────────
async def run_pair_generation(rows: list, out_file) -> tuple:
    sem = asyncio.Semaphore(MAX_SEMAPHORES)
    tasks = [asyncio.ensure_future(_generate_one_pair(sem, row)) for row in rows]

    ok_count = fail_count = 0
    pbar = tqdm(total=len(tasks), desc=f"Pairs (≤{MAX_SEMAPHORES} concurrent)")
    for coro in asyncio.as_completed(tasks):
        record = await coro
        out_file.write(json.dumps(record) + "\n")
        out_file.flush()       # ← safe to interrupt at any point
        if record["status"] == "ok":
            ok_count += 1
        else:
            fail_count += 1
        pbar.update(1)
    pbar.close()
    return ok_count, fail_count

print(f"Async pipeline defined — {MAX_SEMAPHORES} semaphores, incremental flush per record.")

In [ ]:
# ── Section 4: SINGLE-ROW TEST — validates pipeline before full batch ─────────
test_row = all_rows[1]  # Django Developer / Python

print(f"Testing on: [{test_row['jd_index']}] {test_row['job_title']} — '{test_row['skill']}'")
print(f"Anchor    : {test_row['requirement']}\n")

async def _test():
    sem = asyncio.Semaphore(1)
    return await _generate_one_pair(sem, test_row)

test_result = asyncio.run(_test())
print(f"Status       : {test_result['status']}")
print(f"Anchor       : {test_result['anchor']}")
print(f"Positive     : {test_result['positive']}")
print(f"Hard Negative: {test_result['hard_negative']}")

if test_result["status"] == "ok":
    print("\n✓ SUCCESS — safe to launch the full run below.")
else:
    print("\n✗ FAILED — check LLM connectivity before running the full batch.")

In [ ]:
# ── Section 5: FULL BATCH RUN — only run after test above passes ──────────────
print(f"Pending rows to generate pairs for: {len(pending_rows):,}")

if pending_rows:
    with open(PAIRS_FILE, "a", encoding="utf-8") as out_f:
        ok, fail = asyncio.run(run_pair_generation(pending_rows, out_f))
    print(f"\nDone.  OK: {ok:,}  |  Failed: {fail:,}")
else:
    print("All rows already processed — nothing to do.")

print(f"Output: {PAIRS_FILE}")

In [ ]:
# ── Section 6: Inspect & QA Generated Pairs ──────────────────────────────────
pairs_rows_ok = []
pairs_rows_fail = []

with open(PAIRS_FILE, "r", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        if rec["status"] == "ok":
            pairs_rows_ok.append(rec)
        else:
            pairs_rows_fail.append(rec)

pairs_df = pd.DataFrame(pairs_rows_ok)

print(f"Total pairs generated : {len(pairs_rows_ok):,}")
print(f"Failed rows           : {len(pairs_rows_fail):,}")
if pairs_rows_ok or pairs_rows_fail:
    total = len(pairs_rows_ok) + len(pairs_rows_fail)
    print(f"Success rate          : {len(pairs_rows_ok)/total*100:.1f}%")
print(f"Unique JDs covered    : {pairs_df['jd_index'].nunique():,}")
print(f"\nLevel distribution:")
print(pairs_df["level"].value_counts().to_string())

# Show a sample triplet
print("\n─── Sample Pair ────────────────────────────────────────────────────────")
if not pairs_df.empty:
    s = pairs_df.sample(1).iloc[0]
    print(f"Job   : {s['job_title']}  [{s['level']}]")
    print(f"Skill : {s['skill']}")
    print(f"Anchor       : {s['anchor']}")
    print(f"Positive     : {s['positive']}")
    print(f"Hard Negative: {s['hard_negative']}")

In [ ]:
# ── Section 7: Build Training Triplets ───────────────────────────────────────
# Triplet format: (Anchor=JD requirement, Positive=good resume bullet, Negative=bad resume bullet)
# TripletLoss will push Anchor↔Positive together and Anchor↔HardNeg apart.

from sentence_transformers import InputExample
import random

random.seed(42)

examples = []
skipped  = 0

for rec in pairs_rows_ok:
    anchor   = rec.get("anchor", "").strip()
    positive = rec.get("positive", "").strip()
    neg      = rec.get("hard_negative", "").strip()

    if not anchor or not positive or not neg:
        skipped += 1
        continue
    if len(anchor) < 10 or len(positive) < 10 or len(neg) < 10:
        skipped += 1
        continue

    examples.append(InputExample(texts=[anchor, positive, neg]))

random.shuffle(examples)

# 90 / 10 train/val split
split = int(len(examples) * 0.9)
train_examples = examples[:split]
val_examples   = examples[split:]

print(f"Total valid triplets : {len(examples):,}  (skipped {skipped:,} incomplete)")
print(f"Training set         : {len(train_examples):,}")
print(f"Validation set       : {len(val_examples):,}")

# Preview
ex = train_examples[0]
print("\n─── Example Triplet ──────────────────────────────────────────────────────")
print(f"Anchor  : {ex.texts[0]}")
print(f"Positive: {ex.texts[1]}")
print(f"Negative: {ex.texts[2]}")

In [ ]:
# ── Section 8: Load Base Model & Configure Training ───────────────────────────
from sentence_transformers import SentenceTransformer, losses, evaluation
from torch.utils.data import DataLoader

# Same base model used by the production embedding_client
BASE_MODEL = "intfloat/multilingual-e5-small"
print(f"Loading base model: {BASE_MODEL}")
model = SentenceTransformer(BASE_MODEL)

TRAIN_BATCH_SIZE = 32
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=TRAIN_BATCH_SIZE)

# TripletLoss: pushes (anchor↔positive) together, (anchor↔negative) apart
train_loss = losses.TripletLoss(
    model=model,
    distance_metric=losses.TripletDistanceMetric.COSINE,
    triplet_margin=0.5,
)

# Evaluator on val set
val_anchors   = [ex.texts[0] for ex in val_examples]
val_positives = [ex.texts[1] for ex in val_examples]
val_negatives = [ex.texts[2] for ex in val_examples]

evaluator = evaluation.TripletEvaluator(
    anchors=val_anchors,
    positives=val_positives,
    negatives=val_negatives,
    name="confit-val",
)

print(f"Train batches : {len(train_dataloader):,}")
print(f"Val triplets  : {len(val_examples):,}")
print("Model, loss, and evaluator ready.")

In [ ]:
# ── Section 9: Fine-Tune ConFit Model ────────────────────────────────────────
NUM_EPOCHS   = 2
WARMUP_STEPS = max(1, len(train_dataloader) // 10)  # 10% warmup

print(f"Starting training — {NUM_EPOCHS} epochs, batch={TRAIN_BATCH_SIZE}, warmup={WARMUP_STEPS} steps")
print(f"Output: {MODEL_OUTPUT_DIR}")

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=evaluator,
    epochs=NUM_EPOCHS,
    warmup_steps=WARMUP_STEPS,
    evaluation_steps=max(1, len(train_dataloader) // 2),  # eval at midpoint + end of each epoch
    output_path=str(MODEL_OUTPUT_DIR),
    save_best_model=True,
    show_progress_bar=True,
)

print("\n✓ Training complete. Best model saved to:", MODEL_OUTPUT_DIR)

In [ ]:
# ── Section 10: Evaluate Base vs Fine-Tuned Model ────────────────────────────
from sentence_transformers import util

print("Loading fine-tuned model for comparison...")
ft_model   = SentenceTransformer(str(MODEL_OUTPUT_DIR))
base_model = SentenceTransformer(BASE_MODEL)

N_EVAL = min(100, len(val_examples))
eval_sample = val_examples[:N_EVAL]

def triplet_accuracy(m, examples):
    """% of triplets where cos_sim(anchor, positive) > cos_sim(anchor, negative)."""
    a_emb = m.encode([e.texts[0] for e in examples], convert_to_tensor=True, show_progress_bar=False)
    p_emb = m.encode([e.texts[1] for e in examples], convert_to_tensor=True, show_progress_bar=False)
    n_emb = m.encode([e.texts[2] for e in examples], convert_to_tensor=True, show_progress_bar=False)
    pos_sim = util.cos_sim(a_emb, p_emb).diagonal()
    neg_sim = util.cos_sim(a_emb, n_emb).diagonal()
    return (pos_sim > neg_sim).float().mean().item() * 100

base_acc = triplet_accuracy(base_model, eval_sample)
ft_acc   = triplet_accuracy(ft_model,   eval_sample)

print("=" * 55)
print("TRIPLET ACCURACY — anchor closer to positive than negative")
print("=" * 55)
print(f"Base model  ({BASE_MODEL[-25:]}) : {base_acc:.1f}%")
print(f"Fine-tuned  (confit-v1)          : {ft_acc:.1f}%")
print(f"Improvement                      : +{ft_acc - base_acc:.1f}%")

print("\n─── Qualitative Sample (fine-tuned model) ───────────────────────────────")
for ex in eval_sample[:3]:
    a = ft_model.encode(ex.texts[0], convert_to_tensor=True)
    p = ft_model.encode(ex.texts[1], convert_to_tensor=True)
    n = ft_model.encode(ex.texts[2], convert_to_tensor=True)
    ps = util.cos_sim(a, p).item()
    ns = util.cos_sim(a, n).item()
    print(f"  Anchor  : {ex.texts[0][:90]}")
    print(f"  Pos sim : {ps:.3f}  |  Neg sim: {ns:.3f}  |  Gap: {ps-ns:+.3f}")
    print()

In [ ]:
# ── Final Summary ─────────────────────────────────────────────────────────────
print("=" * 60)
print("NB3 COMPLETE — Summary")
print("=" * 60)
print(f"JD requirements used as anchors : {len(all_rows):,}")
print(f"Synthetic pairs generated        : {len(pairs_rows_ok):,}")
print(f"Training triplets                : {len(train_examples):,}")
print(f"Validation triplets              : {len(val_examples):,}")
print(f"Fine-tuned model triplet acc.    : {ft_acc:.1f}%  (base: {base_acc:.1f}%)")
print(f"\nFiles produced:")
print(f"  {PAIRS_FILE}")
print(f"  {MODEL_OUTPUT_DIR}/")
print()
print("Next step: update app/clients/embedding_client.py to load from:")
print(f"  {MODEL_OUTPUT_DIR}")